# WasteSort reproducible training

This notebook downloads real TrashNet images, audits them, trains the four proposal models plus introductory exercises and a weighting ablation, and saves actual results.

In Colab choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. GPU availability is managed by Colab. Training can run on CPU but takes longer.

Every execution creates a new experiment folder. The published study is never overwritten. Scores may vary across hardware and library builds; the dataset split and source hashes are recorded. Group members must inspect the dataset and understand the results before submission.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, tempfile
workspace = Path(tempfile.mkdtemp(prefix="wastesort-"))
source = workspace / "source"
subprocess.run(["git", "clone", "https://github.com/haerinfan/wastesort.git", str(source)], check=True)
source_commit = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", source_commit)
print("Workspace:", workspace)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(source / "requirements.txt")], check=True)


## Prepare a fresh experiment
The source archive and split are checked against the published provenance. Conflicting exact duplicates are excluded; candidate near-duplicates stay in the same partition.

In [ ]:
run = workspace / "experiment"
run.mkdir()
(run / "data").mkdir()
(run / "reports").mkdir()
shutil.copyfile(source / "protocol.json", run / "protocol.json")
for relative in ["data/provenance.json", "data/splits.csv", "reports/data_audit.json"]:
    shutil.copyfile(source / relative, run / relative)
env = dict(os.environ, WASTESORT_ROOT=str(run))
def execute(module):
    subprocess.run([sys.executable, "-u", "-m", "wastesort." + module], cwd=source, env=env, check=True)
execute("data")
(run / "reports/source_commit.txt").write_text(source_commit)


In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(run / "reports/split_counts.csv"))
display(pd.read_csv(run / "reports/excluded.csv"))
execute("figures")
display(Image(filename=str(run / "reports/class_distribution.png")))
display(Image(filename=str(run / "reports/training_examples.png")))


## Train on training data and select using validation only

Primary models: HOG + SVM, HOG + Random Forest, small CNN, MobileNetV2.
Auxiliary experiments: KNN, Decision Tree and weighted small CNN.

The configuration in `protocol.json` is fixed before this cell. Do not edit settings after seeing the final test results and still call that test unseen.

In [ ]:
execute("train")
import json
deployment = json.loads((run / "models/deployment.json").read_text())
print("Model selected BEFORE test evaluation:", deployment["model_id"])
print("Validation macro F1:", deployment["validation_macro_f1"])


## Final held-out evaluation
This runs once after model selection. Bootstrap intervals are descriptive and conditional on a single fitted run. They do not establish campus performance or variation across training seeds.

In [ ]:
execute("evaluate")
execute("exercises")
execute("figures")
display(pd.read_csv(run / "reports/comparison.csv"))
display(Image(filename=str(run / f"reports/{deployment['model_id']}_confusion.png")))
display(Image(filename=str(run / "reports/error_examples.png")))
print((run / "reports/uncertainty.json").read_text())
print("Separate regression exercise:")
print((run / "reports/regression_exercise.json").read_text())


## Save the experiment evidence
The exported ZIP contains reports, checkpoints, protocol and manifests. Raw photographs remain available through the original download source. The notebook itself must also be saved with its outputs for your contribution record.

In [ ]:
export = workspace / "export"
export.mkdir()
for folder in ["reports", "models"]:
    shutil.copytree(run / folder, export / folder, ignore=shutil.ignore_patterns("audit_images"))
(export / "data").mkdir()
for relative in ["data/provenance.json", "data/splits.csv", "protocol.json"]:
    shutil.copyfile(run / relative, export / relative)
archive = shutil.make_archive(str(workspace / "WasteSort_experiment"), "zip", export)
print("Experiment archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download the ZIP from the notebook file browser.")


## Group review

- Verify the dataset audit, candidate duplicates, reference list and actual member contributions.
- Explain validation versus test, macro F1, minority-class recall and the confusion matrix.
- Confirm campus rules before adding bin instructions.
- Report campus photos separately from the TrashNet benchmark.

Dataset: [TrashNet by Gary Thung and Mindy Yang](https://github.com/garythung/trashnet).
AI assisted code and experiment execution; human verification remains required.
